<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_1704%2B%2B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q triton trl evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 49.9 MB/s eta 0:00:00


In [12]:
import re
import torch
import random
import numpy as np
import pandas as pd
import time
import evaluate
from typing import Optional
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftConfig
)
from tqdm import tqdm

In [3]:
@dataclass
class RAGExperiment:
    dataset_name: str = 'phatvo/hotpotqa-raft'
    oracle_presence_ratio: float = 1.0
    sample_size: int = 10
    test_size_ratio: float = 0.2
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    peft_config: Optional[PeftConfig] = field(default_factory=lambda: LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    warmup_steps: int = 5
    seed: int = 1


    def __post_init__(self):
        random.seed(self.seed)
        self.squad_metric = evaluate.load("squad")


    def prepare_data(self):
        dataset: DatasetDict = load_dataset('phatvo/hotpotqa-raft')


        def presence_oracle(example):
            return example['oracle_context'] in example['instruction']


        dataset_with_oracle: Dataset = dataset['train'].filter(lambda ex: presence_oracle(ex))
        dataset_without_oracle: Dataset = dataset['train'].filter(lambda ex: not presence_oracle(ex))

        num_with: int = round(self.sample_size * self.oracle_presence_ratio)
        num_without: int = self.sample_size - num_with

        indices_with: list = random.sample(range(len(dataset_with_oracle)), num_with)
        indices_without: list = random.sample(range(len(dataset_without_oracle)), num_without)

        if indices_with:
            idx_train_with_oracle, idx_test_with_oracle = train_test_split(
                indices_with,
                test_size=self.test_size_ratio,
                shuffle=False,
                random_state=self.seed
            )
        else:
            idx_train_with_oracle, idx_test_with_oracle = [], []

        if indices_without:
            idx_train_without_oracle, idx_test_without_oracle = train_test_split(
                indices_without,
                test_size=self.test_size_ratio,
                shuffle=False,
                random_state=self.seed
            )
        else:
            idx_train_without_oracle, idx_test_without_oracle = [], []

        def get_shuffle_dataset(ids_with, ids_without):
            parts = []
            if ids_with:
                parts.append(dataset_with_oracle.select(ids_with))
            if ids_without:
                parts.append(dataset_without_oracle.select(ids_without))
            if parts:
                return concatenate_datasets(parts).shuffle(seed=self.seed)
            else:
                return Dataset.from_dict({})


        self.train_dataset: Dataset = get_shuffle_dataset(idx_train_with_oracle, idx_train_without_oracle)
        self.test_dataset: Dataset = get_shuffle_dataset(idx_test_with_oracle, idx_test_without_oracle)


    def prepare_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16,
            device_map='auto'
        )
        self.metrics["trainable_params_m"] = self.model.num_parameters(only_trainable=True) / 1e6

    def prepare_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )


    def format_data(self, dataset):
        """Форматируем данные в prompt/completion для SFTTrainer."""
        def format_sft_example(example):
            prompt_messages = [{"role": "user", "content": example["instruction"]}]
            prompt = self.tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            completion = example["cot_answer"]
            return {"prompt": prompt, "completion": completion}

        return dataset.map(
            format_sft_example,
            remove_columns=dataset.column_names
        )


    def add_peft_if_exist(self):
        self.model = get_peft_model(self.model, self.peft_config) if self.peft_config else self.model


    def prepare_training(self):
        self.training_args = SFTConfig(
            output_dir="./qwen-raft-lora",
            num_train_epochs=self.num_train_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            warmup_steps=self.warmup_steps,
            logging_steps=10,
            save_steps=100,
            save_total_limit=2,
            bf16=True,
            report_to="none",
            remove_unused_columns=False,

            max_length=1280,
            packing=False,
            completion_only_loss=True,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.formatted_train_dataset,
            processing_class=self.tokenizer,
        )


    def train(self):
        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()

        self.trainer.train()

        end_time = time.time()
        self.metrics["training_time_min"] = round((end_time - start_time) / 60.0, 2)
        if torch.cuda.is_available():
            self.metrics["peak_memory_gb"] = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
        else:
            self.metrics["peak_memory_gb"] = 0.0

    def extract_answer(self, text: str) -> Optional[str]:
        """
        Извлекает ответ после <ANSWER>:.
        Возвращает None, если маркер отсутствует.
        """
        match = re.search(r"<ANSWER>:\s*(.*?)(?:\n|$)", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        return None


    def evaluate_f1(self, debug_examples: int = 3):
        """
        Вычисляет среднюю F1-меру по тестовому набору, используя метрику SQuAD.
        Если в сгенерированном тексте нет тега <ANSWER>:, F1 = 0.
        """
        self.model.eval()
        formatted_test = self.format_data(self.test_dataset)

        f1_scores = []
        # Списки для метрики evaluate (она может принимать батчи, но мы будем добавлять по одному)
        predictions = []
        references = []

        gen_start = time.time()

        for i, example in enumerate(tqdm(formatted_test, desc="Evaluating F1")):
            prompt = example["prompt"]
            true_completion = example["completion"]
            true_answer = self.extract_answer(true_completion)
            if true_answer is None:
                continue

            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=2048,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            generated_part = full_output[len(prompt):].strip()
            pred_answer = self.extract_answer(generated_part)

            # Отладочный вывод первых примеров
            if i < debug_examples:
                print("\n" + "="*80)
                print(f"📋 ПРИМЕР {i}")
                print("="*80)
                print("\n🔹 ЭТАЛОННЫЙ ОТВЕТ:", true_answer)
                print("🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ:", pred_answer if pred_answer else "❌ ТЕГ НЕ НАЙДЕН")
                print("🔸 Наличие тега <ANSWER> в генерации:", "✅" if pred_answer is not None else "❌")

            if pred_answer is None:
                f1 = 0.0
            else:
                # Формат для метрики squad: predictions — список словарей с ключом 'prediction_text'
                # references — список словарей с ключом 'answers', внутри которого 'text' (список строк)
                pred_dict = {"prediction_text": pred_answer, "id": str(i)}
                ref_dict = {"answers": {"answer_start": [0], "text": [true_answer]}, "id": str(i)}
                # Метрика ожидает списки
                result = self.squad_metric.compute(predictions=[pred_dict], references=[ref_dict])
                f1 = result["f1"] / 100.0   # squad возвращает проценты (0-100)

            f1_scores.append(f1)

        self.metrics["generation_time_min"] = round((time.time() - gen_start) / 60.0, 2)

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0
        print("\n" + "="*80)
        print(f"✅ Средняя F1 по всем {len(f1_scores)} примерам: {avg_f1:.4f}")
        print("="*80)

        self.metrics["f1_squad"] = round(avg_f1, 4)
        return avg_f1


    def evaluate_perplexity(self):
        pass


    def run(self):
        self.prepare_data()
        self.prepare_model()
        self.prepare_tokenizer()
        self.formatted_train_dataset = self.format_data(self.train_dataset)
        self.add_peft_if_exist()
        self.prepare_training()
        self.train()
        self.evaluate_f1()
        self.evaluate_perplexity()
        torch.cuda.empty_cache()

In [4]:
exp = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=200,
    test_size_ratio=0.03,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [5]:
exp.run()

README.md:   0%|          | 0.00/651 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1855 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: Paul McCartney
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1:  50%|█████     | 3/6 [00:15<00:17,  5.99s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: Thomas II de Berkeley
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1: 100%|██████████| 6/6 [00:34<00:00,  5.80s/it]


✅ Средняя F1 по всем 6 примерам: 0.0000


In [6]:
exp2 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=400,
    test_size_ratio=0.03,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [7]:
exp2.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/388 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/388 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/388 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: Devon
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1:  25%|██▌       | 3/12 [00:17<00:53,  5.93s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: February 26, 1982
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1: 100%|██████████| 12/12 [01:06<00:00,  5.54s/it]


✅ Средняя F1 по всем 12 примерам: 0.0000


In [8]:
exp3 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=200,
    test_size_ratio=0.03,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [9]:
exp3.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/194 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/194 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: Paul McCartney
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1:  50%|█████     | 3/6 [00:33<00:37, 12.50s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: Thomas II de Berkeley
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1: 100%|██████████| 6/6 [00:56<00:00,  9.45s/it]


✅ Средняя F1 по всем 6 примерам: 0.0000


In [14]:
exp4 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=400,
    test_size_ratio=0.03,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [11]:
exp4.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/388 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/388 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/388 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: Devon
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1:  25%|██▌       | 3/12 [00:24<01:12,  8.04s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: February 26, 1982
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1: 100%|██████████| 12/12 [01:16<00:00,  6.34s/it]


✅ Средняя F1 по всем 12 примерам: 0.4131


In [17]:
exp5 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=500,
    test_size_ratio=0.1,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [18]:
exp5.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.111383
20,1.046395
30,0.948247
40,0.924203
50,0.830047
60,0.791083
70,0.748441
80,0.730488
90,0.677534
100,0.668662


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Evaluating F1:   2%|▏         | 1/50 [00:07<06:01,  7.38s/it]

...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: 2
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: ❌ ТЕГ НЕ НАЙДЕН
🔸 Наличие тега <ANSWER> в генерации: ❌


Evaluating F1:   6%|▌         | 3/50 [00:17<04:31,  5.78s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: "My Own Worst Enemy"
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: My Own Worst Enemy
🔸 Наличие тега <ANSWER> в генерации: ✅


Evaluating F1: 100%|██████████| 50/50 [06:00<00:00,  7.21s/it]


✅ Средняя F1 по всем 50 примерам: 0.3776


In [19]:
exp6 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=1000,
    test_size_ratio=0.05,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [20]:
exp6.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: 1972
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: 1972
🔸 Наличие тега <ANSWER> в генерации: ✅


Evaluating F1:   6%|▌         | 3/50 [00:19<04:53,  6.25s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: 63,901 acres
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: 63,901 acres
🔸 Наличие тега <ANSWER> в генерации: ✅


Evaluating F1: 100%|██████████| 50/50 [06:29<00:00,  7.79s/it]


✅ Средняя F1 по всем 50 примерам: 0.6441


In [24]:
exp7 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=500,
    test_size_ratio=0.1,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [25]:
exp7.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: 2
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: 2
🔸 Наличие тега <ANSWER> в генерации: ✅


Evaluating F1:   6%|▌         | 3/50 [00:18<04:50,  6.17s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: "My Own Worst Enemy"
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: My Own Worst Enemy
🔸 Наличие тега <ANSWER> в генерации: ✅


Evaluating F1: 100%|██████████| 50/50 [06:26<00:00,  7.73s/it]


✅ Средняя F1 по всем 50 примерам: 0.5802


In [26]:
exp8 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=1000,
    test_size_ratio=0.05,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [27]:
exp8.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.087105
20,1.030178
30,0.982426
40,0.882760
50,0.837005
60,0.804384
70,0.725798
80,0.703626
90,0.652404
100,0.626005


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Evaluating F1:   2%|▏         | 1/50 [00:07<06:06,  7.49s/it]

...


📋 ПРИМЕР 1

🔹 ЭТАЛОННЫЙ ОТВЕТ: 1972
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: 1972
🔸 Наличие тега <ANSWER> в генерации: ✅


Evaluating F1:   6%|▌         | 3/50 [00:19<05:06,  6.52s/it]


📋 ПРИМЕР 2

🔹 ЭТАЛОННЫЙ ОТВЕТ: 63,901 acres
🔹 СГЕНЕРИРОВАННЫЙ ОТВЕТ: 63,901 acres
🔸 Наличие тега <ANSWER> в генерации: ✅


Evaluating F1: 100%|██████████| 50/50 [06:49<00:00,  8.19s/it]


✅ Средняя F1 по всем 50 примерам: 0.6822
